In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "out_data"
OUT_DIR.mkdir(exist_ok=True)

print("ROOT   :", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)


In [2]:
X_FILE = OUT_DIR / "binary_vosa_XPcoeff_110d_L2.csv"
if not X_FILE.exists():
    raise FileNotFoundError(
        f"Missing {X_FILE}\n"
        "This should be created by the data-construction notebook."
    )

df_all = pd.read_csv(X_FILE)

# Validate columns
if "y" not in df_all.columns:
    raise KeyError(f"Missing 'y' column. Columns: {list(df_all.columns)[:20]} ...")

feat_cols = [c for c in df_all.columns if c.startswith("c")]
if len(feat_cols) != 110:
    raise ValueError(f"Expected 110 features c000..c109; got {len(feat_cols)}")

df_all["source_id"] = df_all["source_id"].astype("int64")
df_all["y"] = df_all["y"].astype(int)

print("Loaded:", X_FILE.name)
print("Shape :", df_all.shape)
print("Positives:", int(df_all["y"].sum()), "| Negatives:", int((1 - df_all["y"]).sum()))
df_all.head()


Loaded: binary_vosa_XPcoeff_110d_L2.csv
Shape : (2815, 112)
Positives: 558 | Negatives: 2257


,source_id,y,c000,c001,c002,c003,c004,c005,c006,c007,...,c100,c101,c102,c103,c104,c105,c106,c107,c108,c109
0,1792620565667968,0,0.799447,-0.447756,0.144183,-0.075735,0.035157,0.034266,0.000626,0.009668,...,0.000387,-0.000244,-0.000614,-0.000013,-0.000117,0.000098,0.000188,-0.000206,0.000249,0.000004
1,6052403489630720,0,0.798218,-0.454831,0.147778,-0.076139,0.038208,0.037110,-0.001446,0.011269,...,-0.000082,0.000076,0.000032,-0.000043,0.000010,-0.000005,0.000078,0.000051,-0.000003,-0.000006
2,10844075163628928,0,0.808262,-0.355471,0.090168,-0.042227,0.023493,0.021350,0.002948,0.003520,...,-0.000557,-0.000250,0.000301,-0.000507,-0.000030,-0.000339,-0.000108,-0.000093,0.000037,-0.000006
3,11015044926034816,0,0.807816,-0.389829,0.111283,-0.053689,0.027280,0.025773,0.001740,0.004259,...,0.000629,0.000201,-0.000444,0.000666,0.000048,-0.000311,0.000255,-0.000386,0.000042,-0.000029
4,11963171843658240,0,0.807543,-0.354601,0.091216,-0.047242,0.023972,0.030070,0.000633,0.006451,...,0.000083,-0.000318,-0.000263,0.000088,-0.000089,-0.000048,0.000026,-0.000110,0.000004,-0.000053


In [3]:
X = df_all[feat_cols].to_numpy(dtype=float)
y = df_all["y"].to_numpy(dtype=int)

# 60/20/20 split (robust + common)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE
)

print("Train:", X_tr.shape, "pos:", y_tr.sum(), "neg:", (1-y_tr).sum())
print("Val  :", X_va.shape, "pos:", y_va.sum(), "neg:", (1-y_va).sum())
print("Test :", X_te.shape, "pos:", y_te.sum(), "neg:", (1-y_te).sum())


Train: (1689, 110) pos: 335 neg: 1354
Val  : (563, 110) pos: 112 neg: 451
Test : (563, 110) pos: 111 neg: 452


In [4]:
from sklearn.metrics import confusion_matrix

def _metrics_from_probs(y_true, p_pos, thr):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos).astype(float)

    y_pred = (p_pos >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
    f1   = (2 * prec * sens) / (prec + sens) if (prec + sens) else 0.0
    youden = sens + spec - 1.0
    return sens, spec, prec, acc, f1, youden


def pick_threshold(y_true, p_pos, criterion="youden", grid_size=200):
    """
    Pick threshold on a validation set by either:
      - criterion="youden"  -> maximize Sens+Spec-1
      - criterion="f1"      -> maximize F1
    Uses quantiles of p_pos to define a compact threshold grid.
    """
    p_pos = np.asarray(p_pos).astype(float)

    qs = np.linspace(0.0, 1.0, grid_size)
    thr_grid = np.unique(np.quantile(p_pos, qs))
    thr_grid = np.clip(thr_grid, 0.0, 1.0)

    best = None
    for thr in thr_grid:
        sens, spec, prec, acc, f1, youden = _metrics_from_probs(y_true, p_pos, thr)
        score = youden if criterion == "youden" else f1
        row = (score, float(thr), sens, spec, prec, acc)
        if best is None or row[0] > best[0]:
            best = row

    score, thr, sens, spec, prec, acc = best
    return {"thr": thr, "sens": sens, "spec": spec, "prec": prec, "acc": acc, "score": score}


def style_table_viridis(df):
    """
    Style tables consistently across notebooks:
      - gradient over key metric columns (includes probability metrics if present)
      - fixed formatting for numeric columns
    """
    metric_cols = [c for c in [
        "Sensitivity", "Specificity", "Precision", "Accuracy",
        "ROC AUC", "PR AUC", "Brier", "Log loss"
    ] if c in df.columns]

    # format rules
    fmt = {}
    for c in metric_cols:
        fmt[c] = "{:.3f}"
    if "Threshold" in df.columns:
        fmt["Threshold"] = "{:.3f}"
    if "Best C" in df.columns:
        fmt["Best C"] = "{:.4g}"

    # RF-specific params (ignore if not present)
    if "Best n_estimators" in df.columns:
        fmt["Best n_estimators"] = "{:d}"
    if "Best min_samples_leaf" in df.columns:
        fmt["Best min_samples_leaf"] = "{:d}"
    if "Best min_samples_split" in df.columns:
        fmt["Best min_samples_split"] = "{:d}"
    if "Best max_depth" in df.columns:
        fmt["Best max_depth"] = "{}"
    if "Best max_features" in df.columns:
        fmt["Best max_features"] = "{}"
    if "Best max_leaf_nodes" in df.columns:
        fmt["Best max_leaf_nodes"] = "{}"

    return (df.style
            .format(fmt, na_rep="")
            .background_gradient(subset=metric_cols, cmap="viridis")
            .hide(axis="index"))

print("Helpers ready.")


Helpers ready.


In [5]:
# ---- Grids ----
N_EST_GRID = [300, 700]
MAX_DEPTH_GRID = [None, 15, 25]
MIN_LEAF_GRID = [1, 2, 5]
MIN_SPLIT_GRID = [2, 10, 30]
MAX_FEAT_GRID = ["sqrt", "log2", 0.2, 0.33, 0.5]
MAX_LEAF_NODES_GRID = [None, 100, 300]
BOOTSTRAP_GRID = [True]

# ---- Variants (principles-based) ----
VARIANTS = [
    {"name": "RF (default)",             "class_weight": None,                "calibration": None},
    {"name": "RF (balanced)",            "class_weight": "balanced",          "calibration": None},
    {"name": "RF (balanced_subsample)",  "class_weight": "balanced_subsample","calibration": None},

    # Calibration variants (keep class_weight None unless you explicitly want both)
    {"name": "RF + sigmoid calib",       "class_weight": None,                "calibration": "sigmoid"},
    {"name": "RF + isotonic calib",      "class_weight": None,                "calibration": "isotonic"},
]

print("Variants:", [v["name"] for v in VARIANTS])
print("Grid sizes:",
      len(N_EST_GRID), len(MAX_DEPTH_GRID), len(MIN_LEAF_GRID),
      len(MIN_SPLIT_GRID), len(MAX_FEAT_GRID), len(MAX_LEAF_NODES_GRID), len(BOOTSTRAP_GRID))


Variants: ['RF (default)', 'RF (balanced)', 'RF (balanced_subsample)', 'RF + sigmoid calib', 'RF + isotonic calib']
Grid sizes: 2 3 3 3 5 3 1


In [6]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

def fit_rf(params, variant):
    base = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        min_samples_split=params["min_samples_split"],
        max_features=params["max_features"],
        max_leaf_nodes=params["max_leaf_nodes"],
        bootstrap=params["bootstrap"],
        class_weight=variant["class_weight"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    # Fit base or calibrated
    if variant["calibration"] is None:
        base.fit(X_tr, y_tr)
        return base

    # Calibrate using TRAIN only (internal CV), then threshold tuned on VAL
    cal = CalibratedClassifierCV(base, method=variant["calibration"], cv=3)
    cal.fit(X_tr, y_tr)
    return cal


def search_on_val(variant, criterion):
    best = None
    best_model = None
    best_params = None
    best_thr = None

    for n_estimators in N_EST_GRID:
        for max_depth in MAX_DEPTH_GRID:
            for min_leaf in MIN_LEAF_GRID:
                for min_split in MIN_SPLIT_GRID:
                    for max_feat in MAX_FEAT_GRID:
                        for max_leaf_nodes in MAX_LEAF_NODES_GRID:
                            for bootstrap in BOOTSTRAP_GRID:

                                params = {
                                    "n_estimators": int(n_estimators),
                                    "max_depth": max_depth,
                                    "min_samples_leaf": int(min_leaf),
                                    "min_samples_split": int(min_split),
                                    "max_features": max_feat,
                                    "max_leaf_nodes": max_leaf_nodes,
                                    "bootstrap": bootstrap,
                                }

                                model = fit_rf(params, variant)
                                p_va = model.predict_proba(X_va)[:, 1]
                                pick = pick_threshold(y_va, p_va, criterion=criterion)

                                score = pick["score"]
                                if best is None or score > best:
                                    best = score
                                    best_model = model
                                    best_params = params
                                    best_thr = pick["thr"]

    return {"model": best_model, "params": best_params, "thr": best_thr}


def evaluate_threshold_metrics_on_test(model, thr):
    p_te = model.predict_proba(X_te)[:, 1]
    sens, spec, prec, acc, f1, youden = _metrics_from_probs(y_te, p_te, thr)
    return {"Sensitivity": sens, "Specificity": spec, "Precision": prec, "Accuracy": acc}


def evaluate_probability_metrics_on_test(model):
    """
    Threshold-free probability metrics on TEST (paper-style):
      - ROC AUC
      - PR AUC (Average Precision)
      - Brier score
      - Log loss
    """
    p_te = model.predict_proba(X_te)[:, 1]

    eps = 1e-15
    p_clip = np.clip(p_te, eps, 1 - eps)

    return {
        "ROC AUC": float(roc_auc_score(y_te, p_te)),
        "PR AUC":  float(average_precision_score(y_te, p_te)),
        "Brier":   float(brier_score_loss(y_te, p_te)),
        "Log loss": float(log_loss(y_te, p_clip)),
    }


def run_selection_and_make_tables(criterion):
    """
    Returns:
      - df_test_thr: threshold metrics table (Sensitivity/Specificity/Precision/Accuracy)
      - df_test_prob: probability metrics table (ROC AUC/PR AUC/Brier/Log loss)
    """
    rows_thr = []
    rows_prob = []

    for v in VARIANTS:
        sel = search_on_val(v, criterion=criterion)
        params = sel["params"]
        thr = float(sel["thr"])
        model = sel["model"]

        met_thr = evaluate_threshold_metrics_on_test(model, thr)
        met_prob = evaluate_probability_metrics_on_test(model)

        base_cols = {
            "Variant": v["name"],
            "Best n_estimators": params["n_estimators"],
            "Best max_depth": params["max_depth"],
            "Best min_samples_leaf": params["min_samples_leaf"],
            "Best min_samples_split": params["min_samples_split"],
            "Best max_features": params["max_features"],
            "Best max_leaf_nodes": params["max_leaf_nodes"],
        }

        rows_thr.append({
            **base_cols,
            "Threshold": thr,
            **met_thr
        })

        rows_prob.append({
            **base_cols,
            **met_prob
        })

    return pd.DataFrame(rows_thr), pd.DataFrame(rows_prob)


# ---- run both criteria and display ----
df_test_youden, df_prob_youden = run_selection_and_make_tables("youden")
df_test_f1,     df_prob_f1     = run_selection_and_make_tables("f1")

print("TEST — evaluated using VAL-picked (hyperparams, threshold) from Youden selection:")
display(style_table_viridis(df_test_youden))

print("\nTEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:")
display(style_table_viridis(df_test_f1))

print("\nTEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) for Youden-picked models:")
display(style_table_viridis(df_prob_youden))

print("\nTEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) for F1-picked models:")
display(style_table_viridis(df_prob_f1))


TEST — evaluated using VAL-picked (hyperparams, threshold) from Youden selection:


Variant,Best n_estimators,Best max_depth,Best min_samples_leaf,Best min_samples_split,Best max_features,Best max_leaf_nodes,Threshold,Sensitivity,Specificity,Precision,Accuracy
RF (default),300,,2,30,sqrt,100.0,0.341,0.820,0.958,0.827,0.931
RF (balanced),700,,1,2,0.2,,0.214,0.856,0.936,0.766,0.920
RF (balanced_subsample),700,15.0,1,2,0.2,,0.207,0.856,0.938,0.772,0.922
RF + sigmoid calib,300,,1,10,sqrt,,0.297,0.811,0.954,0.811,0.925
RF + isotonic calib,700,,2,2,sqrt,100.0,0.241,0.820,0.951,0.805,0.925



TEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:


Variant,Best n_estimators,Best max_depth,Best min_samples_leaf,Best min_samples_split,Best max_features,Best max_leaf_nodes,Threshold,Sensitivity,Specificity,Precision,Accuracy
RF (default),300,,1,2,0.33,100.0,0.542,0.766,0.976,0.885,0.934
RF (balanced),300,,1,2,0.2,,0.496,0.757,0.982,0.913,0.938
RF (balanced_subsample),300,,2,10,0.2,100.0,0.666,0.703,0.987,0.929,0.931
RF + sigmoid calib,300,,2,2,0.2,,0.483,0.775,0.978,0.896,0.938
RF + isotonic calib,700,,2,30,0.33,,0.755,0.748,0.987,0.933,0.940



TEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) for Youden-picked models:


Variant,Best n_estimators,Best max_depth,Best min_samples_leaf,Best min_samples_split,Best max_features,Best max_leaf_nodes,ROC AUC,PR AUC,Brier,Log loss
RF (default),300,,2,30,sqrt,100.0,0.927,0.861,0.062,0.237
RF (balanced),700,,1,2,0.2,,0.937,0.869,0.053,0.209
RF (balanced_subsample),700,15.0,1,2,0.2,,0.937,0.870,0.054,0.210
RF + sigmoid calib,300,,1,10,sqrt,,0.932,0.867,0.053,0.204
RF + isotonic calib,700,,2,2,sqrt,100.0,0.930,0.854,0.053,0.261



TEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) for F1-picked models:


Variant,Best n_estimators,Best max_depth,Best min_samples_leaf,Best min_samples_split,Best max_features,Best max_leaf_nodes,ROC AUC,PR AUC,Brier,Log loss
RF (default),300,,1,2,0.33,100.0,0.934,0.871,0.055,0.266
RF (balanced),300,,1,2,0.2,,0.936,0.866,0.054,0.213
RF (balanced_subsample),300,,2,10,0.2,100.0,0.936,0.865,0.055,0.218
RF + sigmoid calib,300,,2,2,0.2,,0.930,0.866,0.051,0.198
RF + isotonic calib,700,,2,30,0.33,,0.926,0.852,0.051,0.309
